# InsightRAG — NVIDIA Embedding & Retrieval Research

**Author:** Mithun Roy — AI and Machine Learning Engineer

This notebook documents the research & development process behind **InsightRAG**, the
retrieval-augmented question-answering engine that powers `main.py`.

It reproduces the exact pipeline used in production, built entirely on **NVIDIA AI Endpoints**:

| Stage | Component | Model / Library |
|-------|-----------|-----------------|
| Document loading | `requests` + `trafilatura` | Robust article extraction |
| Chunking | `RecursiveCharacterTextSplitter` | LangChain Text Splitters |
| Embeddings | `NVIDIAEmbeddings` | `NV-Embed-QA` |
| Vector store | `FAISS` | FAISS (CPU) |
| Generation | `ChatNVIDIA` | `meta/llama3-8b-instruct` |
| Orchestration | `RetrievalQAWithSourcesChain` | LangChain |

The goal: turn a set of news article URLs into a searchable knowledge base and answer
natural-language questions with cited sources.

## 0. Environment setup

InsightRAG authenticates to NVIDIA AI Endpoints via an API key. Store it in a `.env` file
at the project root as `NVIDIA_API_KEY=...` (never hard-code secrets in the notebook).
The key is loaded into the environment with `python-dotenv`.

In [ ]:
import os
import time
import pickle

import requests
import trafilatura
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQAWithSourcesChain

# Loads NVIDIA_API_KEY from .env into the environment
load_dotenv()
assert os.getenv("NVIDIA_API_KEY"), "NVIDIA_API_KEY not found — add it to your .env file"
print("Environment ready — NVIDIA_API_KEY detected.")

## 1. Load data from article URLs

Many news sites either block naive scrapers (HTTP 403) or ship megabytes of navigation
markup that generic HTML parsers choke on. InsightRAG therefore fetches each page with a
browser `User-Agent` via `requests`, then extracts just the main article body with
**`trafilatura`** — fast, and resilient to page bloat. Failed URLs are skipped and reported
rather than crashing the pipeline.

In [ ]:
BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}


def load_articles(urls, timeout=20):
    """Fetch each URL and extract its main article text into Documents."""
    documents, failures = [], []
    for url in urls:
        try:
            resp = requests.get(url, headers=BROWSER_HEADERS, timeout=timeout)
            if resp.status_code != 200:
                failures.append((url, f"HTTP {resp.status_code}"))
                continue
            text = trafilatura.extract(resp.text, include_comments=False, include_tables=False)
            if not text or not text.strip():
                failures.append((url, "no extractable article text"))
                continue
            documents.append(Document(page_content=text.strip(), metadata={"source": url}))
        except Exception as exc:
            failures.append((url, type(exc).__name__))
    return documents, failures


urls = [
    "https://economictimes.indiatimes.com/tech/artificial-intelligence/wayve-courts-automakers-with-ai-driving-system-that-learns-like-humans/articleshow/132112344.cms",
    "https://timesofindia.indiatimes.com/business/india-business/hdfc-bank-shares-drop-2-on-reports-of-probe-regarding-rs-45-crore-interest-payments-bank-strongly-rejects-claims/articleshow/131344798.cms",
]

data, failures = load_articles(urls)
print(f"Loaded {len(data)} documents, {len(failures)} failed")
for url, reason in failures:
    print(f"  skipped {url} -> {reason}")
data[0].metadata

## 2. Split documents into chunks

Long articles exceed the context window of both the embedding and chat models, and smaller
chunks yield more precise retrieval. We split recursively on paragraph, line, sentence, and
comma boundaries with a `chunk_size` of 1000 characters — matching the production config in
`main.py`.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", ","],
    chunk_size=1000,
)

docs = text_splitter.split_documents(data)
print(f"Split into {len(docs)} chunks")
print(docs[0].page_content[:500])

## 3. Generate NVIDIA embeddings

We embed each chunk with NVIDIA's **`NV-Embed-QA`** model via `NVIDIAEmbeddings`. This
model is optimised for question-answering retrieval, producing dense vectors well suited to
semantic similarity search. Below we inspect the embedding dimensionality on a single chunk
before embedding the full corpus.

In [ ]:
embeddings = NVIDIAEmbeddings(model="NV-Embed-QA")

# Sanity check: embed one chunk and inspect the vector dimension
sample_vector = embeddings.embed_query(docs[0].page_content)
print(f"Embedding dimension: {len(sample_vector)}")

## 4. Build the FAISS vector store

`FAISS.from_documents` embeds every chunk and indexes the resulting vectors for fast
approximate nearest-neighbour search. A quick similarity search confirms the index returns
relevant chunks before we wire up the LLM.

In [ ]:
vector_store = FAISS.from_documents(docs, embeddings)
print(f"Indexed {vector_store.index.ntotal} vectors")

# Quick retrieval sanity check
hits = vector_store.similarity_search("What is the price of Punch iCNG?", k=2)
for i, h in enumerate(hits, 1):
    print(f"--- Result {i} ({h.metadata.get('source')}) ---")
    print(h.page_content[:200], "\n")

## 5. Persist the index

The production app caches the FAISS index to disk so it can be reused across sessions
without re-embedding. `main.py` writes the index to `faiss_store.pkl`.

In [ ]:
file_path = "faiss_store.pkl"

with open(file_path, "wb") as f:
    pickle.dump(vector_store, f)

# Reload to verify round-trip serialization
with open(file_path, "rb") as f:
    vector_store = pickle.load(f)
print("Index persisted and reloaded successfully.")

## 6. Retrieval-augmented generation with NVIDIA LLM

Finally we initialise the **`meta/llama3-8b-instruct`** chat model via `ChatNVIDIA` and wrap
it in a `RetrievalQAWithSourcesChain`. For each question the chain retrieves the most
relevant chunks from FAISS, feeds them to the LLM as context, and returns an answer together
with the source URLs it relied on.

In [ ]:
llm = ChatNVIDIA(model="meta/llama3-8b-instruct", temperature=0.9, max_tokens=500)

chain = RetrievalQAWithSourcesChain.from_llm(llm=llm, retriever=vector_store.as_retriever())

query = "What is the price of Punch iCNG?"
result = chain({"question": query}, return_only_outputs=True)

print("ANSWER:\n", result["answer"])
print("\nSOURCES:\n", result.get("sources", ""))

## Conclusion

This notebook validated every stage of the InsightRAG pipeline end-to-end on NVIDIA AI
Endpoints:

1. **Load** article URLs → `Document` objects.
2. **Split** into 1000-character chunks.
3. **Embed** with `NV-Embed-QA`.
4. **Index** in FAISS and persist to disk.
5. **Answer** questions via `meta/llama3-8b-instruct` with cited sources.

The same components — configured identically — run in `main.py` behind the Streamlit UI.